In [1]:
import sqlite3

In [ ]:
conn = sqlite3.connect("/home/boyesh/akoya_pcf/akoya.db")
cursor = conn.cursor()
try:
    cursor.execute("DROP TABLE IF EXISTS pipeline_status_new")
except:
    pass

cursor.execute("""create table pipeline_status_new(
    status_id integer primary key autoincrement, --Primary key, autoincrement
    slide_id integer references slides(slide_id) not null, --Foreign key (slides.slide_id)
    ingestion text, --Not started/In progress/Complete/Failed
    preprocessing_qc text, --Not started/In progress/Complete/Failed
    segmentation text, --Not started/In progress/Complete/Failed
    feature_extraction text, --Not started/In progress/Complete/Failed
    anndata_export text, --Not started/In progress/Complete/Failed
    phenotyping text, --Not started/In progress/Complete/Failed
    spatial_analysis text, --Not started/In progress/Complete/Failed
    last_update datetime, --Timestamp of most recent status change
    notes text --Flag failures and QC concerns
);""")
print("new table made")

cursor.execute("""INSERT INTO pipeline_status_new
SELECT
    status_id,
    slide_id,
    ingestion,
    preprocessing_qc,
    segmentation,
    feature_extraction,
    'Not Started' AS anndata_export,
    phenotyping,
    spatial_analysis,
    last_update,
    notes
FROM pipeline_status;""")
print("data migrated")

cursor.execute("drop table pipeline_status;")
print("old table dropped")

cursor.execute("alter table pipeline_status_new RENAME TO pipeline_status")
print("table renamed")

conn.commit()
conn.close()

new table made
data migrated
old table dropped
table renamed


In [6]:
conn = sqlite3.connect("/home/boyesh/akoya_pcf/akoya.db")
cursor = conn.cursor()

cursor.execute("Select * from pipeline_status limit 1")
print(cursor.fetchall())

conn.close()

[(1, 1, 'Not Started', 'Failed', 'Passed', 'Passed', 'Not started', 'Not Started', 'Not Started', None, None)]


In [7]:
conn = sqlite3.connect("/home/boyesh/akoya_pcf/akoya.db")
cursor = conn.cursor()

cursor.execute("Update pipeline_status set ingestion = 'Complete'")
cursor.execute("Update pipeline_status set anndata_export = 'Not Started'")

conn.commit()
conn.close()

In [3]:
conn = sqlite3.connect("/home/boyesh/akoya_pcf/akoya.db")
cursor = conn.cursor()

cursor.execute("select * from pipeline_status")
print(cursor.fetchall())

cursor.execute("select count(*) from cell_features")
print(cursor.fetchall())

cursor.execute("select count(*) from cell_intensity")
print(cursor.fetchall())

cursor.execute("select distinct slide_id, channel_name from channel_stats")
print(cursor.fetchall())

conn.close()

[(1, 1, 'Complete', 'Failed', 'Passed', 'Passed', 'Not Started', 'Not Started', 'Not Started', None, None), (2, 2, 'Complete', 'Failed', 'Passed', 'Passed', 'Not Started', 'Not Started', 'Not Started', None, None), (3, 3, 'Complete', 'Complete', 'Passed', 'Passed', 'Not Started', 'Not Started', 'Not Started', None, None), (4, 4, 'Complete', 'Complete', 'Passed', 'Passed', 'Not Started', 'Not Started', 'Not Started', None, None), (5, 5, 'Complete', 'Failed', 'Passed', 'Passed', 'Not Started', 'Not Started', 'Not Started', None, None)]
[(3064976,)]
[(24519808,)]
[(1, 'DAPI'), (1, 'Opal 480'), (1, 'Opal 520'), (1, 'Opal 570'), (1, 'Opal 780'), (1, 'Opal 620'), (1, 'Opal 690'), (1, 'Sample AF'), (2, 'DAPI'), (2, 'Opal 480'), (2, 'Opal 520'), (2, 'Opal 570'), (2, 'Opal 780'), (2, 'Opal 620'), (2, 'Opal 690'), (2, 'Sample AF'), (3, 'DAPI'), (3, 'Opal 480'), (3, 'Opal 520'), (3, 'Opal 570'), (3, 'Opal 780'), (3, 'Opal 620'), (3, 'Opal 690'), (3, 'Sample AF'), (4, 'DAPI'), (4, 'Opal 480'), (4,

In [4]:
### Fixing bug in preprocessing
conn = sqlite3.connect("/home/boyesh/akoya_pcf/akoya.db")
cursor = conn.cursor()

cursor.execute("DELETE FROM channel_stats WHERE slide_id IN (1, 2, 3, 4, 5)")
cursor.execute("UPDATE pipeline_status SET preprocessing_qc = 'Not Started'")

conn.commit()
conn.close(
    
)

In [2]:
conn = sqlite3.connect("/home/boyesh/akoya_pcf/akoya.db")
cursor = conn.cursor()

cursor.execute("DELETE FROM channel_stats;")
cursor.execute("UPDATE pipeline_status SET preprocessing_qc = 'Not Started';")

conn.commit()
conn.close()

In [6]:
import anndata
import os

ISILON_BASE = os.environ.get("AKOYA_ISILON")
DB_PATH = os.environ.get("AKOYA_DB")

In [7]:
adata = anndata.read_h5ad(f"{ISILON_BASE}/prototype/anndata/slide_1_052024 P7HuP120 #03 SG03.h5ad")

In [8]:
print(adata)

AnnData object with n_obs × n_vars = 174349 × 8
    obs: 'slide_id', 'label', 'area', 'centroid_x', 'centroid_y', 'eccentricity', 'perimeter', 'solidity', 'slide_name'
    var: 'min_intensity', 'max_intensity', 'mean_intensity', 'nonzero_fraction', 'flagged', 'flag_message'
    obsm: 'spatial'
    layers: 'max_intensity'


In [9]:
print(adata.obs.head())

         slide_id  label    area   centroid_x   centroid_y  eccentricity  \
cell_id                                                                    
1               1      1  1576.0  4385.670685    25.427665      0.521049   
2               1      2    44.0  8941.045455  1494.477273      0.941961   
3               1      3    26.0  8906.653846  1549.269231      0.602197   
4               1      4    45.0  8822.600000  1552.511111      0.695019   
5               1      5    41.0  8829.926829  1613.414634      0.696741   

          perimeter  solidity                slide_name  
cell_id                                                  
1        173.781746  0.869757  052024 P7HuP120 #03 SG03  
2         27.970563  0.830189  052024 P7HuP120 #03 SG03  
3         16.242641  1.000000  052024 P7HuP120 #03 SG03  
4         22.485281  0.978261  052024 P7HuP120 #03 SG03  
5         22.727922  0.931818  052024 P7HuP120 #03 SG03  
